# EVOO Price Dynamics: Reported Thesis Results

This notebook presents selected result tables from the Master's thesis and creates portfolio-ready visualisations. **It does not refit the original forecasting models**, because the original notebook and raw monthly datasets were not included with the supplied PDF.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'


## 1. Recent chronological holdout

The thesis reports that the persistence benchmark achieved the lowest MAE in the recent test period. Lower MAE indicates smaller average absolute forecast errors.

In [ ]:
holdout = pd.read_csv(DATA / 'holdout_metrics.csv').sort_values('MAE_EUR_per_100kg')
display(holdout)
ax = holdout.sort_values('MAE_EUR_per_100kg', ascending=True).plot.barh(
    x='model', y='MAE_EUR_per_100kg', legend=False, figsize=(9, 5),
    title='Recent chronological holdout: MAE by model')
ax.set_xlabel('MAE (€ per 100 kg)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## 2. Expanding-window backtest

The expanding-window evaluation generates sequential one-month-ahead forecasts while preserving time order. The persistence benchmark again achieved the lowest overall MAE.

In [ ]:
expanding = pd.read_csv(DATA / 'expanding_window_metrics.csv').sort_values('MAE_EUR_per_100kg')
display(expanding)
ax = expanding.sort_values('MAE_EUR_per_100kg', ascending=True).plot.barh(
    x='model', y='MAE_EUR_per_100kg', legend=False, figsize=(9, 5),
    title='Expanding-window backtest: MAE by model')
ax.set_xlabel('MAE (€ per 100 kg)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## 3. Forecasting performance by price regime

Ridge B slightly outperformed the persistence benchmark in the high-price regime. The low-price regime contains only eight out-of-sample observations, so that comparison should be interpreted cautiously.

In [ ]:
regime = pd.read_csv(DATA / 'regime_mae.csv')
display(regime)
ax = regime.set_index('model')[['Low_MAE', 'Medium_MAE', 'High_MAE']].plot.bar(
    figsize=(10, 5), title='Expanding-window MAE by price regime')
ax.set_ylabel('MAE (€ per 100 kg)')
ax.set_xlabel('')
ax.legend(['Low', 'Medium', 'High'], title='Price regime')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()


## 4. XGBoost C: reported SHAP importance

The current EVOO price was the dominant predictor in XGBoost C. SHAP values explain how a fitted model uses its features; they do not establish causal relationships. This is a reported importance table, not a new SHAP computation.

In [ ]:
shap = pd.read_csv(DATA / 'xgboost_c_shap.csv').sort_values('mean_absolute_SHAP')
display(shap.sort_values('mean_absolute_SHAP', ascending=False))
ax = shap.plot.barh(x='feature', y='mean_absolute_SHAP', legend=False, figsize=(9, 7),
                    title='XGBoost C: reported mean absolute SHAP values')
ax.set_xlabel('Mean absolute SHAP value')
ax.set_ylabel('')
plt.tight_layout()
plt.show()


## 5. Main business takeaway

The persistence benchmark was strongest overall, so extra variables and model complexity did not systematically improve one-month-ahead forecasts. Operational indicators added a modest benefit for Ridge B during high-price conditions. The next step toward full reproducibility is to add the original Jupyter notebook and the permitted source datasets, then rerun all preprocessing, model fitting, chronological validation and SHAP analysis.